In [1]:
import os
import pandas as pd
import pickle

from string import punctuation
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer

MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

eng_stopword = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [2]:
def getTag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'
    
def lemmatizing(token):
    lemmatized = []
    tagged = pos_tag(token)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, getTag(tag))
        lemmatized.append(result)
    return lemmatized

def preprocessing(sentence):
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in eng_stopword]
    tokens = [token for token in tokens if token not in punctuation]
    tokens = [token for token in tokens if token.isalpha()]
    tokens = lemmatizing(tokens)
    return tokens

df = pd.read_csv(DATA_PATH)
x = df['Statement']
y = df['Sentiment']

allSentence = ' '.join(x)
allTokens = preprocessing(allSentence)
freq_dist = FreqDist(allTokens)
print(freq_dist)

def extract_feature(sentence):
    featured = {}
    for word in freq_dist.keys():
        featured[word] = (word in sentence)
    return featured

feature_set = [(extract_feature(preprocessing(sentence)),sentiment)
               for sentence, sentiment in zip(x, y)
]

split = int(len(feature_set) * 0.8)
train_data = feature_set[:split]
test_data = feature_set[split:]

def trainModel():
    model = NaiveBayesClassifier.train(train_data)
    model.show_most_informative_features(7)

    acc = accuracy(model, test_data)
    print(f"Accuracy: {acc}")

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
    return model

<FreqDist with 5035 samples and 28353 outcomes>


In [3]:
def show_pos_tag(sentence):
    tokens = preprocessing(sentence)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word}: {tag}")
    return tokens

def show_syn_ant(tokens):
    for token in tokens:
        syno = []
        anto = []
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                syno.append(lemma.name())
                for ant in lemma.antonyms():
                    anto.append(ant.name())
        
        if syno:
            print(f"Syn: {syno}")
        else:
            print("No syn")

        if anto:
            print(f"Anto: {anto}")
        else:
            print("No ant")

def predictStatement(model, tokens):
    feature = extract_feature(tokens)
    predicition = model.classify(feature)
    print(f"Prediction: {predicition}")

def analyzeStatement(model, sentence):
    tokens = show_pos_tag(sentence)
    show_syn_ant(tokens)
    predictStatement(model, tokens)

In [4]:
sentence = ''

while True:
    print("Input: ")
    print("1. Input statement")
    print("2. Analyze statement")
    print("3. Exit")
    choice = input(">> ")
    if choice == '3':
        break
    elif choice == '1':
        sentence = input("Input statement: ")
        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, 'rb') as f:
                model = pickle.load(f)
        else:
            model = trainModel()
    elif choice == '2':
        analyzeStatement(model, sentence)

Input: 
1. Input statement
2. Analyze statement
3. Exit
Input: 
1. Input statement
2. Analyze statement
3. Exit
budiono: NN
siegar: NN
No syn
No ant
No syn
No ant
Prediction: positive
Input: 
1. Input statement
2. Analyze statement
3. Exit
